# Hybrid pre-processing (Nico)

**Question:** `models_ludovico` §3 found that neither recipe wins everywhere: count encoding helps naive Bayes, one-hot helps KNN, and for logistic regression, decision tree and boosting the two are tied. Does a **hybrid** of the two recipes beat V03?

**Recipes compared** (all with the month of purchase, as in V03):

| Name | Numeric columns | Categorical columns |
|---|---|---|
| **L = V03** | median, signed log, robust scaler (Ludovico) | one-hot, levels under 50 rows grouped |
| H1 | as L | **count** for `transport_method` (80 levels), one-hot for the others |
| H2 | as L | **count** for every original categorical, one-hot for the month |
| H3 | **iterative imputation** (Nico), signed log, robust scaler | one-hot |
| H4 | iterative, signed log, robust scaler | count for `transport_method`, one-hot for the others |
| N | Nico's recipe: iterative, log on the 3 day intervals only, standard scaler | count for every categorical |

**Count encoding is written here with scikit-learn only** (`CountEncoder` below): `src/preprocessing.py` uses `category_encoders`, which is not on the list of packages the brief allows.

**Two instruments:** the V03 model (logistic regression + L1 selection, `C = 0.05`) and the tuned gradient boosting of `tuning_ludovico`. **Measures:** ROC AUC on the official folds (paired with V03), honest F1 (threshold from folds 1–2, F1 on fold 3: so far it predicted Kaggle within ±0.002), and ROC AUC on the 8 half-year folds as a second check.

**Rule, declared before the results:** a hybrid replaces V03 only if it is better beyond 2 SEM on the official folds, not worse on the 8 folds, and its honest F1 is not lower.

# Setup
`%run` executes `preprocessing_ludovico.ipynb` (about 2 minutes): clean data, `official_folds`, `prep_folds`, `evaluate`, `best_threshold`.

In [1]:
%%capture
%run ./preprocessing_ludovico.ipynb

In [2]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import IterativeImputer

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

ALWAYS_DROP = ["customer_id", "purchase_date", "random_noise"]
X_full = train.drop(columns=[TARGET])
NUMERIC = X_full.drop(columns=ALWAYS_DROP).select_dtypes("number").columns.tolist()
ORIGINAL_CATEGORICAL = list(CATEGORICAL)            # from cleaning_ludovico
DAY_INTERVALS = ["days_since_previous_purchase", "prior_mean_gap_days", "prior_std_gap_days"]

early = np.concatenate([va for _, va in official_folds[:-1]])
last = official_folds[-1][1]


def add_month(df):
    """The `calendar` group of features_ludovico, then drop the identifiers."""
    df = df.assign(purchase_month=df["purchase_date"].dt.month.astype(str))
    return df.drop(columns=[c for c in ALWAYS_DROP if c in df.columns])


def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))

# 1. The building blocks

In [3]:
class CountEncoder(BaseEstimator, TransformerMixin):
    """Each label becomes its share of the training rows; a label never seen in training becomes 0."""

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.shares_ = [X[c].value_counts(normalize=True) for c in X.columns]
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return np.column_stack([X.iloc[:, j].map(s).fillna(0.0).to_numpy(dtype=float)
                                for j, s in enumerate(self.shares_)])

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features if input_features is not None else [f"x{j}" for j in range(self.n_features_in_)], dtype=object)


class ScaledIterativeImputer(BaseEstimator, TransformerMixin):
    """Iterative imputation on standardised columns, returned in the original units (as in preprocessing_nico)."""

    def fit(self, X, y=None):
        self.scaler_ = StandardScaler().fit(X)
        self.imputer_ = IterativeImputer(max_iter=10, random_state=RANDOM_STATE).fit(self.scaler_.transform(X))
        return self

    def transform(self, X):
        return self.scaler_.inverse_transform(self.imputer_.transform(self.scaler_.transform(X)))

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)


def log_some(columns):
    """Signed log on the given numeric columns only (by position in NUMERIC)."""
    where = [NUMERIC.index(c) for c in columns]

    def apply(X):
        X = np.array(X, dtype=float, copy=True)
        X[:, where] = signed_log1p(X[:, where])
        return X
    return FunctionTransformer(apply, feature_names_out="one-to-one")


def one_hot():
    return OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50, sparse_output=False)


def recipe(numeric="ludovico", counted=(), linear=True):
    """Pre-processing with the numeric plan of `numeric` and count encoding for the categoricals in `counted`."""
    impute = ScaledIterativeImputer() if numeric in ("iterative", "nico") else SimpleImputer(strategy="median")
    if not linear:
        num_steps = [("impute", impute)]
    elif numeric == "nico":
        num_steps = [("impute", impute), ("log", log_some(DAY_INTERVALS)), ("scale", StandardScaler())]
    else:
        num_steps = [("impute", impute), ("log", FunctionTransformer(signed_log1p, feature_names_out="one-to-one")),
                     ("scale", RobustScaler())]
    all_cat = ORIGINAL_CATEGORICAL + ["purchase_month"]
    hot = [c for c in all_cat if c not in counted]
    blocks = [("num", Pipeline(num_steps), NUMERIC)]
    if hot:
        blocks.append(("hot", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("encode", one_hot())]), hot))
    if counted:
        count_steps = [("impute", SimpleImputer(strategy="most_frequent")), ("count", CountEncoder())]
        if linear:
            count_steps.append(("scale", StandardScaler() if numeric == "nico" else RobustScaler()))
        blocks.append(("count", Pipeline(count_steps), list(counted)))
    return ColumnTransformer(blocks)


l1_selector = SelectFromModel(
    make_pipeline(StandardScaler(), LogisticRegression(l1_ratio=1, solver="saga", C=0.05, max_iter=5000)),
    importance_getter="named_steps.logisticregression.coef_",
)
TUNED_BOOSTING = dict(min_samples_leaf=200, max_leaf_nodes=31, max_iter=100, max_features=0.5,
                      learning_rate=0.05, l2_regularization=10.0)  # tuning_ludovico, logged with V05


def v03_model(prep):
    return Pipeline([("features", FunctionTransformer(add_month)), ("prep", prep), ("select", clone(l1_selector)),
                     ("model", LogisticRegression(max_iter=2000))])


def boosting_model(prep):
    return Pipeline([("features", FunctionTransformer(add_month)), ("prep", prep),
                     ("model", HistGradientBoostingClassifier(random_state=RANDOM_STATE, **TUNED_BOOSTING))])


RECIPES_TESTED = {
    "L = V03": dict(),
    "H1: count transport": dict(counted=("transport_method",)),
    "H2: count all original": dict(counted=tuple(ORIGINAL_CATEGORICAL)),
    "H3: iterative": dict(numeric="iterative"),
    "H4: iterative + count transport": dict(numeric="iterative", counted=("transport_method",)),
    "N: Nico's recipe": dict(numeric="nico", counted=tuple(ORIGINAL_CATEGORICAL) + ("purchase_month",)),
}

**Check:** recipe L must give back V03 exactly (ROC AUC 0.7958 on the official folds, `features_ludovico` §12).

In [4]:
v03_scores, _ = evaluate(v03_model(recipe()), official_folds, X_full, y)
assert abs(v03_scores["ROC AUC"].mean() - 0.7958) < 5e-4, v03_scores["ROC AUC"].mean()
print("recipe L reproduces V03:", round(v03_scores["ROC AUC"].mean(), 4))

recipe L reproduces V03: 0.7958


# 2. Results

In [5]:
def paired_vs(a, b):
    d = np.asarray(a) - np.asarray(b)
    sem = d.std(ddof=1) / np.sqrt(len(d))
    return d.mean(), sem, f"{(d > 0).sum()}/{len(d)}"


def board(make, linear=True):
    """Official folds (AUC, honest F1) and 8 half-year folds (AUC) for every recipe, paired with recipe L."""
    official, half_year, rows = {}, {}, {}
    for name, kwargs in RECIPES_TESTED.items():
        scores, oof = evaluate(make(recipe(linear=linear, **kwargs)), official_folds, X_full, y)
        official[name] = scores["ROC AUC"].to_numpy()
        half_year[name] = evaluate(make(recipe(linear=linear, **kwargs)), prep_folds, X_full, y)[0]["ROC AUC"].to_numpy()
        t, _ = best_threshold(y.iloc[early], oof.iloc[early])
        rows[name] = {"ROC AUC": official[name].mean(), "fold 1 / 2 / 3": " / ".join(f"{v:.3f}" for v in official[name]),
                      "honest F1": f1_score(y.iloc[last], oof.iloc[last] >= t), "threshold": t}
    for name in rows:
        diff, sem, won = paired_vs(official[name], official["L = V03"])
        diff8, sem8, won8 = paired_vs(half_year[name], half_year["L = V03"])
        rows[name] |= {"diff vs L": diff, "SEM": sem, "folds won": won,
                       "8 folds: diff vs L": diff8, "8 folds: SEM": sem8, "8 folds won": won8}
    return pd.DataFrame(rows).T


FORMAT = {"ROC AUC": "{:.4f}", "honest F1": "{:.3f}", "threshold": "{:.2f}", "diff vs L": "{:+.4f}", "SEM": "{:.4f}",
          "8 folds: diff vs L": "{:+.4f}", "8 folds: SEM": "{:.4f}"}
linear_board = board(v03_model)
linear_board.style.format(FORMAT)

,ROC AUC,fold 1 / 2 / 3,honest F1,threshold,diff vs L,SEM,folds won,8 folds: diff vs L,8 folds: SEM,8 folds won
L = V03,0.7958,0.779 / 0.807 / 0.801,0.802,0.43,+0.0000,0.0000,0/3,+0.0000,0.0000,0/8
H1: count transport,0.7960,0.779 / 0.803 / 0.805,0.800,0.36,+0.0002,0.0024,2/3,-0.0017,0.0020,4/8
H2: count all original,0.7925,0.775 / 0.795 / 0.807,0.808,0.38,-0.0033,0.0052,1/3,-0.0035,0.0028,4/8
H3: iterative,0.7959,0.781 / 0.805 / 0.801,0.795,0.46,+0.0001,0.0013,2/3,-0.0004,0.0014,4/8
H4: iterative + count transport,0.7963,0.780 / 0.804 / 0.805,0.807,0.42,+0.0005,0.0021,2/3,-0.0020,0.0021,3/8
N: Nico's recipe,0.7906,0.788 / 0.787 / 0.797,0.787,0.42,-0.0051,0.0086,1/3,-0.0060,0.0044,2/8


**Reading (V03 model: logistic regression + L1)**
- **No hybrid changes the ranking:** on the official folds every ROC AUC difference is within ±0.0005 of V03 except two losses (H2 −0.0033, N −0.0051). On the 8 half-year folds **every** variant is slightly below V03 (−0.0004 to −0.0060).
- **Count encoding of `transport_method` (H1, H4)** ties on ROC AUC. Its honest F1 moves (0.800 and 0.807 against 0.802), but only because the threshold chosen on folds 1–2 moves (0.36, 0.42 against 0.43); the ranking itself is not better, and on the 8 folds both lose a little (−0.0017, −0.0020).
- **H2** has the highest honest F1 (0.808) with a *lower* ROC AUC on both fold schemes: a better cut on one fold, not a better model.
- **Iterative imputation (H3)** gives exactly the same ranking as the median (+0.0001): the L1 step and the log already make the model insensitive to how the gaps are filled.
- **Nico's full recipe (N)** is the worst for this model: standard scaling and log on only 3 columns leave the skewed amounts dominating the L1 selection.

In [6]:
boosting_board = board(boosting_model, linear=False)
boosting_board.style.format(FORMAT)

,ROC AUC,fold 1 / 2 / 3,honest F1,threshold,diff vs L,SEM,folds won,8 folds: diff vs L,8 folds: SEM,8 folds won
L = V03,0.8008,0.786 / 0.805 / 0.812,0.799,0.38,+0.0000,0.0000,0/3,+0.0000,0.0000,0/8
H1: count transport,0.7986,0.785 / 0.800 / 0.811,0.794,0.39,-0.0023,0.0011,0/3,-0.0007,0.0016,5/8
H2: count all original,0.7991,0.785 / 0.803 / 0.809,0.799,0.37,-0.0017,0.0009,0/3,+0.0009,0.0025,5/8
H3: iterative,0.7969,0.787 / 0.800 / 0.804,0.801,0.39,-0.0040,0.0027,1/3,-0.0011,0.0018,3/8
H4: iterative + count transport,0.7979,0.788 / 0.799 / 0.806,0.798,0.38,-0.0029,0.0026,1/3,-0.0006,0.0026,4/8
N: Nico's recipe,0.8004,0.795 / 0.801 / 0.805,0.797,0.42,-0.0005,0.0048,1/3,-0.0019,0.0028,4/8


**Reading (tuned gradient boosting)**
- **Recipe L is also the best for the tuned boosting** on the official folds: every hybrid is below it (−0.0005 to −0.0040), H1 on all 3 folds.
- On the 8 folds the differences are mixed and within noise (−0.0019 to +0.0009).
- Trees can already split a one-hot block of rare levels or a count column equally well, and they ignore scaling: the recipe matters little for them, as `models_ludovico` §3 found.

# 3. Decision
The rule of the introduction, applied by code to the V03 model.

In [7]:
passed = [name for name, row in linear_board.iterrows() if name != "L = V03"
          and row["diff vs L"] > 2 * row["SEM"] and row["8 folds: diff vs L"] >= 0
          and row["honest F1"] >= linear_board.loc["L = V03", "honest F1"]]
print("hybrids that pass the rule:", passed or "none")

hybrids that pass the rule: none


# 4. Summary

| Recipe | V03 model: ROC AUC vs V03 (official / 8 folds) | Honest F1 | Tuned boosting: ROC AUC vs L (official) |
|---|---|---|---|
| L = V03 | – | 0.802 | – |
| H1: count transport | +0.0002 / −0.0017 | 0.800 | −0.0023 |
| H2: count all original | −0.0033 / −0.0035 | 0.808 | −0.0017 |
| H3: iterative | +0.0001 / −0.0004 | 0.795 | −0.0040 |
| H4: iterative + count transport | +0.0005 / −0.0020 | 0.807 | −0.0029 |
| N: Nico's recipe | −0.0051 / −0.0060 | 0.787 | −0.0005 |

- **No hybrid passes the rule.** The two recipes and their combinations rank purchases equally well (all within ±0.005 ROC AUC); the small differences in honest F1 come from the threshold chosen on folds 1–2, not from a better ranking.
- **Decision: keep Ludovico's recipe (L) for every model.** It is the simplest, it is the one confirmed on Kaggle, and it is also the best for the tuned boosting.
- **For the report:** the pre-processing is not where the remaining error comes from. Once the columns are log-scaled and the redundant ones removed by L1, how gaps are filled and how labels are encoded hardly change the result.

### Line for `docs/decisions.md`

| Decision | Alternatives considered | Evidence |
|---|---|---|
| Keep Ludovico's pre-processing recipe for every model; no hybrid | count encoding of `transport_method` or of every categorical; iterative imputation; both; Nico's full recipe | `preprocessing_hybrid_nico` §2: V03 model within ±0.0005 ROC AUC or worse on the official folds, slightly worse on all 8-fold checks; tuned boosting worse with every hybrid on the official folds |